# Assignment 7

## Submit as an HTML file

### Print your name below

In [2]:
print("Matthew Dang")

Matthew Dang


### Import the "pandas" "numpy" and "statsmodels.formula.api" libraries

In [3]:
# Write your answer here:
import pandas as pd
import numpy as np
import statsmodels.formula.api as sm


#### In the code chunk below read the CSV file named `results.csv` in the `data` <br> folder and print the first 5 rows of the dataset. Browse the dataset.

In [4]:
results = pd.read_csv("data/results.csv")
print(results.head(5))

   resultId  raceId  driverId  constructorId number  grid position  \
0         1      18         1              1     22     1        1   
1         2      18         2              2      3     5        2   
2         3      18         3              3      7     7        3   
3         4      18         4              4      5    11        4   
4         5      18         5              1     23     3        5   

  positionText  positionOrder  points  laps         time milliseconds  \
0            1              1    10.0    58  1:34:50.616      5690616   
1            2              2     8.0    58       +5.478      5696094   
2            3              3     6.0    58       +8.163      5698779   
3            4              4     5.0    58      +17.181      5707797   
4            5              5     4.0    58      +18.014      5708630   

  fastestLap rank fastestLapTime fastestLapSpeed  statusId  
0         39    2       1:27.452         218.300         1  
1         41    3 

### (a)  Check Column Types and Data Cleaning

- Use the function .dtypes to get the column types
- Identify which columns have data types that might need conversion
- The 'milliseconds' column contains string values that should be numeric. Create a new column called 'race_time_ms' that:
    - Converts the column to a numeric data type
    - Replaces any non-numeric values with NaN

In [5]:
# Write your answer here
results.dtypes
if 'milliseconds' in results.columns:
    results['race_time_ms'] = pd.to_numeric(results['milliseconds'], errors='coerce')
else:
    print("NaN")
print(results[['race_time_ms']].head(11))



    race_time_ms
0      5690616.0
1      5696094.0
2      5698779.0
3      5707797.0
4      5708630.0
5            NaN
6            NaN
7            NaN
8            NaN
9            NaN
10           NaN


### (b) Create Categorical Variables

- Create a new column called 'finish_category' that categorizes the race finish positions as follows:
    - Positions 1-3: 'Podium'
    - Positions 4-10: 'Points'
    - Positions 11-20: 'Midfield'
    - Positions >20: 'Backmarker'

Hint: Use the pd.cut() function

In [6]:
# Write your answer here
results['positionOrder'] = pd.to_numeric(results['positionOrder'], errors='coerce')
results['finish_category'] = pd.cut(
    results['positionOrder'],
    bins=[0, 3, 10, 20, np.inf],
    labels=['Podium', 'Points', 'Midfield', 'Backmarker'],
    right = True
)
print(results[['positionOrder', 'finish_category']])


       positionOrder finish_category
0                  1          Podium
1                  2          Podium
2                  3          Podium
3                  4          Points
4                  5          Points
...              ...             ...
25835             16        Midfield
25836             17        Midfield
25837             18        Midfield
25838             19        Midfield
25839             20        Midfield

[25840 rows x 2 columns]


### (c) Calculate Race Duration
- For rows where 'milliseconds' is available, create a new column <br>
'race_duration_minutes' that converts milliseconds to minutes by dividing <br>
by (1000*60).
- Display the average race duration by 'constructorId' for the top 5 <br>
constructors with the shortest average race times

In [7]:
# Write your answer here
results['race_duration_minutes'] = pd.to_numeric(results['milliseconds'], errors='coerce') / (1000 * 60)
avg_race_duration = results.groupby('constructorId')['race_duration_minutes'].mean()
print(avg_race_duration.head(5))

constructorId
1     97.461541
2     94.119229
3     96.770410
4     95.154676
5    102.491413
Name: race_duration_minutes, dtype: float64


### (d) Driver Performance Analysis

- Calculate the following statistics for each driver, grouped by 'driverId':
    - Average finishing position
    - Total points
    - Number of races completed
    - Best finishing position

- Sort the results by total points in descending order
- Display the top 10 drivers based on total points

In [8]:
# Write your answer here
drivergroup = results.groupby('driverId')['positionOrder'].mean()
drivergroup1 = results.groupby('driverId')['points'].sum()
drivergroup2 = results.groupby('driverId')['laps'].count()
drivergroup3 = results.groupby('driverId')['positionOrder'].min()

driver_stats = pd.DataFrame({
    'avg_finish': drivergroup,
    'total_points': drivergroup1,
    'races_completed': drivergroup2,
    'best_finish': drivergroup3
})
driver_stats_sorted = driver_stats.sort_values(by='total_points', ascending=False)
print(driver_stats_sorted.head(10))

          avg_finish  total_points  races_completed  best_finish
driverId                                                        
1           4.787097        4396.5              310            1
20          7.093333        3098.0              300            1
4           8.494413        2061.0              358            1
830         6.533742        1983.5              163            1
8           8.491477        1873.0              352            1
822         7.601990        1778.0              201            1
3           8.252427        1594.5              206            1
30          6.879870        1566.0              308            1
817         9.883621        1307.0              232            1
18          9.695793        1235.0              309            1


### (e) Linear Regression
Create a linear regression model that predicts 'points' based on 'grid' (starting position) and 'laps' completed <br>
Use the following steps:

- Clean the data to remove any non-numeric values and missing values
- Create the regression formula using smf.ols 
- Display the summary of the regression model using model.summary()

What is the predicted points for a driver starting in position 3 and completing 55 laps?

Hint: Use ```.dropna()''' to remove missing values from the points, grid, and laps <br>
variables.

In [9]:
# Write your answer here
na_omit = results[['points', 'grid', 'laps']].dropna()
linear_model = sm.ols(formula='points ~ grid + laps', data=na_omit).fit()
model_summary = linear_model.summary()
predicted_points = linear_model.predict({'grid': [3], 'laps': [55]})[0]
model_summary, predicted_points




(<class 'statsmodels.iolib.summary.Summary'>
 """
                             OLS Regression Results                            
 Dep. Variable:                 points   R-squared:                       0.215
 Model:                            OLS   Adj. R-squared:                  0.215
 Method:                 Least Squares   F-statistic:                     3530.
 Date:                Mon, 24 Mar 2025   Prob (F-statistic):               0.00
 Time:                        19:06:31   Log-Likelihood:                -70440.
 No. Observations:               25840   AIC:                         1.409e+05
 Df Residuals:                   25837   BIC:                         1.409e+05
 Df Model:                           2                                         
 Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
 ---------------------------------------------------------------------